# Energy Consumption Regression Analysis

This notebook analyzes the energy consumption of various energy types based on the average output tokens per prompt using polynomial regression models. The steps involve loading the data, transforming it, fitting the regression models, predicting values, and visualizing the results.


## 1. Load Data

First, we load the data from a CSV file into a pandas DataFrame.

In [1]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import matplotlib.pyplot as plt

import altair as alt

from IPython.display import display, Math

In [2]:
# Function to load CSV data
def load_csv_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    df = pd.read_csv(file_path)
    return df

In [3]:
# Specify the path to your CSV file
file_path = 'data/input_tok_summary_vllm.csv'

# Load the data into a DataFrame
df_vllm_emission_regression = load_csv_data(file_path)
df_vllm_emission_regression = df_vllm_emission_regression[df_vllm_emission_regression['test_type'] == 'Input-tok-vllm']

# Display the first few rows of the DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,actual_emissions_per_100k_output_tokens,actual_total_energy_per_100k_output_tokens,actual_cpu_energy_per_100k_output_tokens,actual_gpu_energy_per_100k_output_tokens,actual_ram_energy_per_100k_output_tokens,actual_idle_gpu_energy_per_100k_output_tokens,actual_non_idle_gpu_energy_per_100k_output_tokens
0,Input-tok-vllm,llama3,8,100,10000,103.070068,0.103070,792.092229,81641.0,311000.0,81.641,311.0,9.268134,13.938128,1.818787,9.205500,2.913842,3.927713,5.277786
1,Input-tok-vllm,llama3,8,500,10000,254.655488,0.254655,663.618134,168994.0,1253000.0,168.994,1253.0,11.872039,17.854080,2.170529,12.205982,3.477569,4.688104,7.517878
2,Input-tok-vllm,llama3,8,1000,10000,430.458733,0.430459,589.306200,253672.0,2922000.0,253.672,2922.0,13.505549,20.310676,2.444194,13.950415,3.916067,5.279278,8.671137
3,Input-tok-vllm,llama3,8,5000,10000,1805.381947,1.805382,184.195926,332544.0,10089000.0,332.544,10089.0,43.658650,65.657213,7.819408,45.309416,12.528390,16.890228,28.419188


## 2. Data Transformation

Convert energy values from kilowatt-hours (kWh) to watt-hours (Wh) for better granularity, and calculate prompts per second.

In [4]:
# Transform energy values from kWh to Wh

df_vllm_emission_regression['total_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_total_energy_per_100k_output_tokens']
df_vllm_emission_regression['ram_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_ram_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['cpu_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_cpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_idle_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_idle_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['gpu_non_idle_energy_100k_output_tokens_Wh'] = df_vllm_emission_regression['actual_non_idle_gpu_energy_per_100k_output_tokens'] 
df_vllm_emission_regression['prompt_per_sec'] = df_vllm_emission_regression['num_prompts'] / df_vllm_emission_regression['total_time']

df_vllm_emission_regression = df_vllm_emission_regression[['test_type', 
                                                           'model_type', 
                                                           'parameters',
                                                           'num_examples', 
                                                           'num_prompts', 
                                                           'total_out_tok', 
                                                           'total_in_tok', 
                                                           'avg_out_tok', 
                                                           'avg_in_tok', 
                                                           'total_energy_100k_output_tokens_Wh', 
                                                           'ram_energy_100k_output_tokens_Wh', 
                                                           'gpu_energy_100k_output_tokens_Wh', 
                                                           'cpu_energy_100k_output_tokens_Wh']]

# Display the updated DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,total_energy_100k_output_tokens_Wh,ram_energy_100k_output_tokens_Wh,gpu_energy_100k_output_tokens_Wh,cpu_energy_100k_output_tokens_Wh
0,Input-tok-vllm,llama3,8,100,10000,81641.0,311000.0,81.641,311.0,13.938128,2.913842,9.205500,1.818787
1,Input-tok-vllm,llama3,8,500,10000,168994.0,1253000.0,168.994,1253.0,17.854080,3.477569,12.205982,2.170529
2,Input-tok-vllm,llama3,8,1000,10000,253672.0,2922000.0,253.672,2922.0,20.310676,3.916067,13.950415,2.444194
3,Input-tok-vllm,llama3,8,5000,10000,332544.0,10089000.0,332.544,10089.0,65.657213,12.528390,45.309416,7.819408


## 3. Polynomial Regression Model Fitting
Fit polynomial regression models for each type of energy consumption using the average output tokens per prompt as the feature variable.

In [5]:
def fit_polynomial_regression(X, y, degree=2):
    polynomial_features = PolynomialFeatures(degree=degree)
    linear_regression = LinearRegression()
    model = make_pipeline(polynomial_features, linear_regression)
    model.fit(X, y)
    return model

In [6]:
# Define the feature variable
X = df_vllm_emission_regression[['avg_in_tok']]

In [7]:
# Fit models for each energy consumption type
models = {}
energy_types = [
    'total_energy_100k_output_tokens_Wh', 
    'ram_energy_100k_output_tokens_Wh', 
    'gpu_energy_100k_output_tokens_Wh', 
    'cpu_energy_100k_output_tokens_Wh',
]

In [8]:
for energy_type in energy_types:
    y = df_vllm_emission_regression[energy_type]
    models[energy_type] = fit_polynomial_regression(X, y)
    coefs = models[energy_type].named_steps['linearregression'].coef_
    intercept = models[energy_type].named_steps['linearregression'].intercept_
    

## 4. Display Model Coefficients
Display the coefficients of the polynomial regression models for each type of energy consumption.

In [9]:
def display_model_coefficients(model, energy_type):
    coefs = model.named_steps['linearregression'].coef_
    intercept = model.named_steps['linearregression'].intercept_

    # Format the coefficients to 4 decimal places for readability
    coefs = np.round(coefs, 5)
    intercept = np.round(intercept, 5)
    
    
    print("="*20 + f" Regression for {energy_type} " + "="*20 + "\n")
    
    # Print raw coefficients to check their values
    print(f"Raw coefficients:\n intercept={intercept}, coefs={coefs}\n")
    
    print("Formula:")
    # Generate the LaTeX formula
    latex_formula = (
        f"\\hat{{y}} = {intercept:.5f} + {coefs[1]:.5f} x + {coefs[2]:.5f} x^2"
    )

    # Display the LaTeX formula
    display(Math(latex_formula))

    print("\n\n")

In [10]:
# Display coefficients for each energy consumption type
for energy_type in energy_types:
    display_model_coefficients(models[energy_type], energy_type)

==================== Regression for total_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=14.40673, coefs=[0.      0.00109 0.     ]

Formula:


<IPython.core.display.Math object>




==================== Regression for ram_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=2.99942, coefs=[0.     0.0001 0.    ]

Formula:


<IPython.core.display.Math object>




==================== Regression for gpu_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=9.53511, coefs=[0.      0.00092 0.     ]

Formula:


<IPython.core.display.Math object>




==================== Regression for cpu_energy_100k_output_tokens_Wh ====================

Raw coefficients:
 intercept=1.8722, coefs=[0.e+00 6.e-05 0.e+00]

Formula:


<IPython.core.display.Math object>

## 5. Predict Values
Define x values for prediction and predict the corresponding energy consumption values using the fitted models.

In [19]:
# Define the x values for prediction
predicted_values = {'avg_in_tok': [10, 50, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 30000]}

x_values = pd.DataFrame(predicted_values)

# Predict values ensuring feature names are consistent
for energy_type in energy_types:
    model = models[energy_type]
    predicted_values[energy_type] = model.predict(x_values)

# Display the predicted values
predicted_values_df = pd.DataFrame(predicted_values)
predicted_values_df

,avg_in_tok,total_energy_100k_output_tokens_Wh,ram_energy_100k_output_tokens_Wh,gpu_energy_100k_output_tokens_Wh,cpu_energy_100k_output_tokens_Wh
0,10,14.417654,3.000466,9.544338,1.872849
1,50,14.462127,3.004802,9.581772,1.875553
2,250,14.703468,3.030480,9.781419,1.891569
3,500,15.049619,3.071952,10.060229,1.917439
4,1000,15.890170,3.186139,10.715349,1.988682
5,1500,16.928386,3.341986,11.500471,2.085929
6,2000,18.164266,3.539493,12.415594,2.209179
7,3000,21.229019,4.059486,14.635844,2.533689
8,4000,25.084431,4.746118,17.376100,2.962213
9,5000,29.730501,5.599389,20.636362,3.494750


## 6. Combine Actual and Predicted Data
Combine the actual and predicted data into a single DataFrame for visualization.

In [20]:
# Combine actual and predicted data into a single DataFrame
data = []
for energy_type in energy_types:
    for index, row in df_vllm_emission_regression.iterrows():
        data.append({'avg_in_tok': row['avg_in_tok'], 'Energy_Consumption': row[energy_type], 'Type': 'Actual', 'Energy_Type': energy_type})
    for i, x in enumerate(x_values['avg_in_tok']):
        data.append({'avg_in_tok': x, 'Energy_Consumption': predicted_values[energy_type][i], 'Type': 'Predicted', 'Energy_Type': energy_type})

combined_df = pd.DataFrame(data)
combined_df

,avg_in_tok,Energy_Consumption,Type,Energy_Type
0,311.0,13.938128,Actual,total_energy_100k_output_tokens_Wh
1,1253.0,17.854080,Actual,total_energy_100k_output_tokens_Wh
2,2922.0,20.310676,Actual,total_energy_100k_output_tokens_Wh
3,10089.0,65.657213,Actual,total_energy_100k_output_tokens_Wh
4,10.0,14.417654,Predicted,total_energy_100k_output_tokens_Wh
...,...,...,...,...
59,3000.0,2.533689,Predicted,cpu_energy_100k_output_tokens_Wh
60,4000.0,2.962213,Predicted,cpu_energy_100k_output_tokens_Wh
61,5000.0,3.494750,Predicted,cpu_energy_100k_output_tokens_Wh
62,10000.0,7.717637,Predicted,cpu_energy_100k_output_tokens_Wh


## 7. Visualize Results
Create an Altair chart to visualize the actual and predicted energy consumption values.

In [21]:
# Create Altair chart
base = alt.Chart(combined_df[combined_df['Type'] == 'Actual']).mark_point(size=100, filled=True).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'),
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type', 'Type']
).properties(
    width=1200,
    height=600
)


# Highlight predicted values
predicted = alt.Chart(combined_df[combined_df['Type'] == 'Predicted']).mark_point(size=10, filled=False).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'), 
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type']
)

regression = predicted.transform_regression('avg_in_tok', 'Energy_Consumption', groupby=['Energy_Type'], method="quad").mark_line()

# Combine charts
final_chart = base + regression + predicted

# Display the chart in Streamlit
final_chart

alt.LayerChart(...)

In [22]:
import pandas as pd
import numpy as np

def calculate_energy_change(model, start_tokens, end_tokens):
    """
    Calculate the percentage change in energy consumption when changing the number of input tokens.

    Parameters:
    - model: The scikit-learn model used for prediction.
    - start_tokens (int): The initial number of input tokens.
    - end_tokens (int): The new number of input tokens.

    Returns:
    - results (dict): A dictionary containing the token counts, predicted energies, and percentage change range.
    """
    # Create a DataFrame with the start and end token counts
    x_values = pd.DataFrame({'avg_in_tok': [start_tokens, end_tokens]})
    
    # Predict the energy consumption for both token counts
    y_values = model.predict(x_values)
    
    # Extract the predicted energy values
    energy_start = y_values[0]
    energy_end = y_values[1]
    
    # Calculate the percentage change in energy consumption
    energy_change_percent = ((energy_end - energy_start) / energy_start) * 100
    
    # Calculate the factor change in energy consumption
    energy_change_factor = energy_end / energy_start if energy_start != 0 else np.inf
    
    # Calculate the next lower and higher multiples of 50% for percentage change
    lower_percent = np.floor(energy_change_percent / 50) * 50
    upper_percent = np.ceil(energy_change_percent / 50) * 50

    # Calculate the next lower and higher multiples of 0.5x for factor change
    lower_factor = np.floor(energy_change_factor / 0.5) * 0.5
    upper_factor = np.ceil(energy_change_factor / 0.5) * 0.5

    # Handle percentage change range formatting
    if energy_change_percent == 0:
        energy_change_range = "No Change"
    elif energy_change_percent < 0:
        energy_change_range = f"Decrease of {abs(energy_change_percent):.2f}%"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_bound = 10000  # You can adjust this value as needed
        if upper_percent > max_upper_bound:
            energy_change_range = f">{int(max_upper_bound)}%"
        else:
            if lower_percent == upper_percent:
                energy_change_range = f"{int(upper_percent)}%"
            else:
                energy_change_range = f"{int(lower_percent)}% – {int(upper_percent)}%"

    # Handle factor change range formatting
    if energy_change_factor == 1:
        energy_change_factor_range = "No Change"
    elif energy_change_factor < 1:
        energy_change_factor_range = f"{energy_change_factor:.2f}x"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_factor = 1000  # You can adjust this value as needed
        if upper_factor > max_upper_factor:
            energy_change_factor_range = f">{max_upper_factor}x"
        else:
            if lower_factor == upper_factor:
                energy_change_factor_range = f"{upper_factor:.1f}x"
            else:
                energy_change_factor_range = f"{lower_factor:.1f}x – {upper_factor:.1f}x"

    # Prepare the results dictionary
    results = {
        'start_tokens': start_tokens,
        'end_tokens': end_tokens,
        'energy_start': energy_start,
        'energy_end': energy_end,
        'energy_change_percent': energy_change_percent,
        'energy_change_range': energy_change_range,
        'energy_change_factor': energy_change_factor,
        'energy_change_factor_range': energy_change_factor_range
    }
    
    return results


In [23]:
# Define the start and end token counts
start_tokens = 1000
end_tokens = 10000

# Call the function to calculate the energy change
results = calculate_energy_change(models['total_energy_100k_output_tokens_Wh'], start_tokens, end_tokens)

# Print the results in a nicely formatted way
print(f"Energy Consumption Analysis:\n")
print(f"- Start Tokens: {results['start_tokens']}")
print(f"- End Tokens: {results['end_tokens']}\n")
print(f"- Energy Consumption at Start: {results['energy_start']:.2f} Wh")
print(f"- Energy Consumption at End: {results['energy_end']:.2f} Wh\n")
print(f"Percentage Change in Energy Consumption: {results['energy_change_range']}")
print(f"Factor Change in Energy Consumption: {results['energy_change_factor_range']}")


Energy Consumption Analysis:

- Start Tokens: 1000
- End Tokens: 10000

- Energy Consumption at Start: 15.89 Wh
- Energy Consumption at End: 64.82 Wh

Percentage Change in Energy Consumption: 300% – 350%
Factor Change in Energy Consumption: 4.0x – 4.5x
